# Capítulo 5 · Proyecto 1: comparación de modelos

> ¿Qué modelo recomendarías para clasificar la presencia de enfermedad cardiaca y qué resultados respaldan tu elección?

Utiliza el dataset [UCI Heart Disease de Kaggle](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data), archivo `heart_disease_uci.csv`, y documenta tu trabajo en un notebook reproducible.

**Sobre los datos.** El enunciado pide el archivo `heart_disease_uci.csv` de Kaggle, que necesita una cuenta para bajarse. En su lugar armé la tabla con los cuatro archivos públicos de UCI: Cleveland, Hungría, Suiza y VA Long Beach. Tiene las mismas columnas y los mismos nombres, pero **no es el archivo exacto de Kaggle**, así que algunos detalles pueden cambiar. La tabla quedó en `data/heart_disease_uci.csv`, que git ignora, como el resto de los datos.

Cada fila es un paciente, y `dataset` dice de qué hospital viene. `num` va de 0 a 4 y mide qué tan grave es la enfermedad del corazón.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, fbeta_score, make_scorer, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

# Que scikit-learn devuelva DataFrames (con nombres de columna)
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "heart_disease_uci.csv")
df_original = df.copy()          # guardo los datos tal como llegaron, antes de tocar nada
df.head()

## Etapa 1: limpieza y preparación

1. Identifica los valores faltantes y los valores inválidos, decide cómo tratarlos y justifica brevemente tu decisión.
2. Explora los posibles outliers y explica si los conservarás o aplicarás algún tratamiento.
3. Prepara las variables numéricas y categóricas. Crea el objetivo binario (**target**) a partir de `num`: **target = 0 si num = 0** (ausencia de enfermedad) y **target = 1 si num es 1, 2, 3 o 4** (presencia de enfermedad). Excluye tanto `num` como el target de las entradas del modelo.

### 1.1 Primer vistazo

In [ ]:
print("Filas y columnas:", df.shape)

resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "% de nulos": (df.isna().mean() * 100).round(1),
    "valores únicos": df.nunique(),
})
display(resumen)

print("Filas duplicadas (sin contar el id):", df.drop(columns="id").duplicated().sum())
display(df[df.drop(columns="id").duplicated(keep=False)])
display(df[["age", "trestbps", "chol", "thalch", "oldpeak"]].describe().T.round(2))

In [ ]:
categoricas = ["dataset", "sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
for col in categoricas:
    print(df[col].value_counts(dropna=False).rename(col).to_frame().T.to_string(), "\n")

**Qué veo.** Son 920 pacientes y 16 columnas. No faltan datos en `age`, `sex` ni `cp`, pero en el resto sí: `ca` le falta al 66.4% (611 pacientes), `thal` al 52.8% (486) y `slope` al 33.6% (309). Les siguen `fbs` con 9.8%, `oldpeak` con 6.7%, `trestbps` con 6.4%, `thalch` y `exang` con 6.0%, `chol` con 3.3% y `restecg` con 0.2%.

También hay valores que no pueden ser reales: `trestbps` y `chol` tienen mínimo 0, y `oldpeak` tiene un mínimo de -2.6. Lo reviso en la sección siguiente.

Hay 2 filas repetidas sin contar el `id`. Son dos pacientes de Hungría consecutivos (`id` 405 y 406) con los mismos valores. Las conservo, porque sin nombre ni número de expediente no puedo saber si son una captura doble o dos personas distintas, y son 2 de 920.

La mayoría de los pacientes son hombres (726 contra 194 mujeres) y el tipo de dolor más común en el pecho es `asymptomatic` (496). Las cinco variables numéricas con las que voy a trabajar son `age`, `trestbps`, `chol`, `thalch` y `oldpeak`.

### 1.2 Valores faltantes y valores inválidos

In [ ]:
print("chol igual a 0:", (df["chol"] == 0).sum(), "| trestbps igual a 0:", (df["trestbps"] == 0).sum(),
      "| oldpeak negativo:", (df["oldpeak"] < 0).sum(), "| age menor a 18:", (df["age"] < 18).sum())

por_origen = df.drop(columns=["id", "dataset", "num"]).isna().groupby(df["dataset"]).mean().mul(100).round(0).T
display(por_origen)
print("Pacientes con chol igual a 0, por origen:")
display((df["chol"] == 0).groupby(df["dataset"]).sum().rename("chol = 0").to_frame().T)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(por_origen, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% de nulos"}, ax=ax)
ax.set_title("% de valores faltantes por columna y por origen de los datos")
plt.show()

**Qué veo.** Los faltantes no son parejos: dependen del hospital. Cleveland casi no tiene huecos (1% en `ca` y en `thal`). En Hungría falta `ca` en el 99%, `thal` en el 90% y `slope` en el 65%. En Suiza falta `fbs` en el 61% y `ca` en el 96%. En VA Long Beach faltan `trestbps`, `thalch`, `exang` y `oldpeak` en alrededor del 26% al 28%, además de `ca` (99%) y `thal` (83%). Es decir, hay columnas que algunos hospitales simplemente no midieron.

Además hay 172 pacientes con `chol` igual a 0, todos de Suiza (123) y de VA Long Beach (49), y 1 con `trestbps` igual a 0. Un colesterol o una presión de 0 no existe en una persona viva, así que los trato como faltantes mal capturados. Los 12 pacientes con `oldpeak` negativo vienen también de Suiza y de VA Long Beach. No tengo manera de confirmar si son errores, así que los dejo.

**Qué hago con cada tipo de faltante.** Eliminar filas no es posible, porque `ca` falta en 611 de 920. Rellenar con la media no me gusta en `chol` ni en `trestbps`, porque tienen valores extremos. Entonces:

- En las columnas numéricas relleno con la **mediana**, que casi no se mueve con los extremos.
- En las categóricas, el faltante pasa a ser una categoría propia, `faltante`.
- Para `slope`, `thal` y `ca`, que son las que más faltan, no decido todavía: lo hago con evidencia en la sección 2.3.

Después de convertir los ceros, `chol` queda con 202 faltantes (172 más los 30 que ya tenía) y `trestbps` con 60.

Antes de decidir qué hacer con los faltantes, convierto a vacío los ceros que no pueden ser reales y dejo las columnas de sí y no con texto. Son reglas fila por fila que no aprenden nada de los datos, así que no filtran información del test.

In [ ]:
df = df_original.copy()

# Un colesterol o una presión de 0 no existe en una persona viva: son valores faltantes mal capturados
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)

# fbs y exang vienen como True y False: los paso a texto para tratarlas como categorías
for col in ["fbs", "exang"]:
    df[col] = df[col].map({True: "sí", False: "no"})

print("Faltantes por columna después de convertir los ceros:")
display(df.isna().sum().rename("nulos").to_frame().T)

### 1.3 Outliers

In [ ]:
numericas = ["age", "trestbps", "chol", "thalch", "oldpeak"]
filas = []
for col in numericas:
    serie = df[col].dropna()
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    bajo, alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    z = (serie - serie.mean()) / serie.std()
    filas.append({"columna": col, "mínimo": serie.min(), "máximo": serie.max(), "límite bajo (IQR)": round(bajo, 1),
                  "límite alto (IQR)": round(alto, 1), "fuera por IQR": int(((serie < bajo) | (serie > alto)).sum()),
                  "fuera por Z-score (|z| > 3)": int((z.abs() > 3).sum())})
display(pd.DataFrame(filas).set_index("columna"))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, col in zip(axes, numericas):
    sns.boxplot(y=df[col], ax=ax, color="#8172b3")
    ax.set_title(col)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Los casos extremos de cada columna, para decidir si son errores o personas reales
print("oldpeak mayor a 4:", df.loc[df["oldpeak"] > 4, ["dataset", "oldpeak", "num"]].shape[0], "pacientes | de ellos con enfermedad:",
      int((df.loc[df["oldpeak"] > 4, "num"] > 0).sum()))
print("chol mayor a 500:", df.loc[df["chol"] > 500, ["dataset", "chol", "num"]].shape[0], "pacientes | de ellos con enfermedad:",
      int((df.loc[df["chol"] > 500, "num"] > 0).sum()))
print("trestbps mayor a 180:", df.loc[df["trestbps"] > 180, ["dataset", "trestbps", "num"]].shape[0], "pacientes | de ellos con enfermedad:",
      int((df.loc[df["trestbps"] > 180, "num"] > 0).sum()))
print("thalch menor a 80:", df.loc[df["thalch"] < 80, ["dataset", "thalch", "num"]].shape[0], "pacientes | de ellos con enfermedad:",
      int((df.loc[df["thalch"] < 80, "num"] > 0).sum()))
print("oldpeak negativo:", df.loc[df["oldpeak"] < 0, ["dataset", "oldpeak", "num"]].shape[0], "pacientes, todos de:",
      df.loc[df["oldpeak"] < 0, "dataset"].unique().tolist())

**Qué veo.** Por IQR hay valores fuera de los límites en `trestbps` (27), `chol` (23), `oldpeak` (16) y `thalch` (2); `age` no tiene ninguno. Por Z-score con corte de 3 los números bajan a 7 en `trestbps`, 8 en `chol` y 7 en `oldpeak`, y `thalch` y `age` quedan en 0.

Para decidir si son errores o personas reales miro los casos más extremos. De los 6 pacientes con `oldpeak` mayor a 4, 5 tienen enfermedad. De los 4 con `chol` mayor a 500, 3 la tienen. De los 8 con `trestbps` mayor a 180, 7 la tienen, y de los 11 con `thalch` menor a 80, 10 la tienen.

**Decisión: conservarlos.** Son valores posibles (una presión muy alta o una frecuencia cardiaca máxima muy baja) y se concentran justo en los pacientes enfermos, que son los que quiero detectar. Si los borro, pierdo información útil. La regresión logística y KNN se ven más afectados por los extremos porque usan la escala, pero son pocos casos y los estandarizo, y el árbol solo compara contra umbrales.

### 1.4 Variable objetivo y variables de entrada

In [ ]:
df["target"] = (df["num"] > 0).astype(int)       # 0 si num = 0 (sin enfermedad), 1 si num es 1, 2, 3 o 4

display(df["num"].value_counts().sort_index().rename("pacientes").to_frame().T)
conteo = df["target"].value_counts().sort_index()
display(pd.DataFrame({"pacientes": conteo, "porcentaje": (conteo / len(df) * 100).round(1)}).rename(index={0: "0 = sin enfermedad", 1: "1 = con enfermedad"}))

por_origen_target = df.groupby("dataset")["target"].agg(["size", "mean"]).rename(columns={"size": "pacientes", "mean": "% con enfermedad"})
por_origen_target["% con enfermedad"] = (por_origen_target["% con enfermedad"] * 100).round(1)
display(por_origen_target)

# Entradas: todo menos el identificador, num y target
NUMERICAS = ["age", "trestbps", "chol", "thalch", "oldpeak"]
CATEGORICAS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
ORIGEN = ["dataset"]
print("Entradas numéricas:", NUMERICAS)
print("Entradas categóricas:", CATEGORICAS, "| origen de los datos (se decide en la etapa 2):", ORIGEN)

**Qué veo.** Con `target = 1` si `num` es 1, 2, 3 o 4, quedan 411 pacientes sin enfermedad (44.7%) y 509 con enfermedad (55.3%). Las clases están bastante parejas en total, pero no por hospital: tiene enfermedad el 45.9% en Cleveland, el 36.1% en Hungría, el 93.5% en Suiza y el 74.5% en VA Long Beach. El origen de los datos cambia mucho la proporción.

Las entradas del modelo son las 5 numéricas y las 7 categóricas (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope` y `thal`), más `ca` y `dataset`, que decido en la sección 2.3. Quedan fuera `id`, que solo es un identificador, y `num` y `target`, porque son la respuesta.

## Etapa 2: entrenamiento y comparación

1. Separa los datos en entrenamiento, validación y prueba, usando las mismas particiones para todos los modelos y `random_state=42`. Ajusta la imputación y el escalado únicamente con entrenamiento.
2. Entrena **regresión logística, KNN y árbol de decisión**, aplicando escalado cuando corresponda.
3. Usa **GridSearchCV** con validación cruzada sobre los datos de entrenamiento para buscar los mejores hiperparámetros de los tres modelos. Compara las mejores configuraciones en validación y selecciona el modelo con mejor resultado según una métrica justificada.
4. Compara entrenamiento y evaluación para identificar sobreajuste. Presenta una tabla de métricas y las matrices de confusión de prueba.
5. Recomienda un modelo y justifica tu elección con los resultados: qué errores comete, qué ventajas tiene y cuáles son sus limitaciones.

### 2.1 Partición: entrenamiento, validación y prueba

Separo 60% para entrenamiento, 20% para validación y 20% para prueba, con `stratify` y `random_state=42`, y uso las mismas tres partes para todos los modelos. Entrenamiento sirve para ajustar los modelos y buscar sus hiperparámetros con validación cruzada. Validación sirve para elegir entre los tres modelos. Prueba se usa una sola vez, al final, y no cambia ninguna decisión.

In [ ]:
X = df[NUMERICAS + ["ca"] + CATEGORICAS + ORIGEN]
y = df["target"]

X_resto, X_test, y_resto, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_resto, y_resto, test_size=0.25, stratify=y_resto, random_state=SEED)

tamanos = pd.DataFrame({
    "pacientes": [len(y_train), len(y_val), len(y_test)],
    "% con enfermedad": [round(y_train.mean() * 100, 1), round(y_val.mean() * 100, 1), round(y_test.mean() * 100, 1)],
}, index=["entrenamiento", "validación", "prueba"])
display(tamanos)
display(pd.crosstab(X_train["dataset"], "pacientes en entrenamiento").T)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

Quedan 552 pacientes en entrenamiento, 184 en validación y 184 en prueba, y el porcentaje con enfermedad es casi igual en las tres (55.3%, 55.4% y 55.4%). En entrenamiento hay 194 pacientes de Cleveland, 172 de Hungría, 67 de Suiza y 119 de VA Long Beach.

### 2.2 Cómo preparo las variables y cómo mido

Todo el preprocesamiento va dentro de un `Pipeline`, así la mediana, el valor más frecuente y el escalado se calculan solo con los datos de entrenamiento de cada partición. Los números faltantes se rellenan con la mediana. Las categorías faltantes pasan a ser una categoría propia, `faltante`. Las categorías se codifican con `OneHotEncoder`. El escalado, con `StandardScaler`, solo lo uso en regresión logística y KNN, porque el árbol solo compara cada variable contra un umbral.

Como métrica uso **F2**, que mezcla recall y precision pero cuenta el recall el doble. La razón es que aquí el error más grave es decirle sano a un paciente enfermo (un falso negativo). También reporto accuracy, recall, precision y AUC.

In [ ]:
f2 = make_scorer(fbeta_score, beta=2)


def armar_preprocesador(escalar, numericas=NUMERICAS, categoricas=CATEGORICAS):
    pasos_num = [("imputador", SimpleImputer(strategy="median"))]
    if escalar:
        pasos_num.append(("escalador", StandardScaler()))
    pasos_cat = [("imputador", SimpleImputer(strategy="constant", fill_value="faltante")),
                 ("codificador", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
    return ColumnTransformer([("num", Pipeline(pasos_num), numericas), ("cat", Pipeline(pasos_cat), categoricas)])


def armar_modelo(nombre, escalar, parametros=None, categoricas=CATEGORICAS, numericas=NUMERICAS):
    modelos = {
        "Regresión logística": LogisticRegression(max_iter=5000, random_state=SEED),
        "KNN": KNeighborsClassifier(),
        "Árbol de decisión": DecisionTreeClassifier(random_state=SEED),
    }
    modelo = modelos[nombre]
    if parametros:
        modelo.set_params(**parametros)
    return Pipeline([("preprocesador", armar_preprocesador(escalar, numericas, categoricas)), ("modelo", modelo)])


ESCALAR = {"Regresión logística": True, "KNN": True, "Árbol de decisión": False}


def metricas(modelo, X_datos, y_datos):
    pred = modelo.predict(X_datos)
    prob = modelo.predict_proba(X_datos)[:, 1]
    return {"accuracy": accuracy_score(y_datos, pred), "recall": recall_score(y_datos, pred), "precision": precision_score(y_datos, pred, zero_division=0),
            "F2": fbeta_score(y_datos, pred, beta=2), "AUC": roc_auc_score(y_datos, prob)}

### 2.3 Tres decisiones con evidencia (solo con entrenamiento)

Antes de buscar hiperparámetros hay tres cosas que decidir: si dejo `slope` y `thal`, si dejo `ca` (las tres son las columnas con más faltantes) y si `dataset`, el origen de los datos, entra como variable. Comparo cada opción con validación cruzada repetida sobre entrenamiento y con los tres modelos en una versión sencilla. La regla, escrita antes de ver el resultado: la opción más completa solo gana si mejora el F2 promedio de los tres modelos en más de 0.01. Si no, me quedo con la más simple. Con `dataset` no aplico la regla, porque el hospital de origen no es una característica del paciente; solo lo mido para saber cuánto aportaría.

In [ ]:
modelos_base = {
    "Regresión logística": {"C": 1.0},
    "KNN": {"n_neighbors": 15},
    "Árbol de decisión": {"max_depth": 4, "min_samples_leaf": 10},
}
MARGEN = 0.01


def comparar(opciones):
    filas = []
    for etiqueta, (num, cat) in opciones.items():
        fila = {"opción": etiqueta}
        for nombre, parametros in modelos_base.items():
            modelo = armar_modelo(nombre, ESCALAR[nombre], parametros, categoricas=cat, numericas=num)
            fila[nombre] = cross_val_score(modelo, X_train[num + cat], y_train, cv=cv_repetida, scoring=f2).mean()
        filas.append(fila)
    tabla = pd.DataFrame(filas).set_index("opción")
    tabla["promedio"] = tabla.mean(axis=1)
    return tabla


# Decisión 1: slope y thal
BASE_CAT = [c for c in CATEGORICAS if c not in ("slope", "thal")]
tabla_slope_thal = comparar({"A: sin slope ni thal": (NUMERICAS, BASE_CAT),
                             "B: con slope y thal (categoría 'faltante')": (NUMERICAS, CATEGORICAS)})
display(tabla_slope_thal.round(4))
mejora = tabla_slope_thal["promedio"].iloc[1] - tabla_slope_thal["promedio"].iloc[0]
CAT_FINAL = list(CATEGORICAS) if mejora > MARGEN else BASE_CAT
print("Mejora de B sobre A:", round(mejora, 4), "| categóricas que se quedan:", CAT_FINAL)

**Decisión 1: `slope` y `thal`.** Dejar las dos columnas sube el F2 promedio de 0.827 a 0.838, una mejora de 0.0109. Es apenas más que el margen de 0.01 que me puse, así que según mi regla **me quedo con `slope` y `thal`**, con `faltante` como categoría. Aun así la diferencia es chica, y el hecho de que a un hospital le falten esas columnas también delata de dónde viene el paciente, que es una trampa que reviso más adelante.

In [ ]:
# Decisión 2: ca (le falta el 66%)
tabla_ca = comparar({"sin ca": (NUMERICAS, CAT_FINAL), "con ca (mediana si falta)": (NUMERICAS + ["ca"], CAT_FINAL)})
display(tabla_ca.round(4))
mejora_ca = tabla_ca["promedio"].iloc[1] - tabla_ca["promedio"].iloc[0]
NUM_FINAL = NUMERICAS + ["ca"] if mejora_ca > MARGEN else list(NUMERICAS)
print("Mejora de usar ca:", round(mejora_ca, 4), "| numéricas que se quedan:", NUM_FINAL)

**Decisión 2: `ca`.** Agregarla baja el F2 promedio de 0.8380 a 0.8366, es decir, no ayuda (-0.0013). Con tantos faltantes (66%) rellenarla con la mediana es casi inventarla, así que **la dejo fuera**.

In [ ]:
# Decisión 3: el origen de los datos (solo para medir cuánto aportaría)
tabla_origen = comparar({"sin dataset": (NUM_FINAL, CAT_FINAL), "con dataset": (NUM_FINAL, CAT_FINAL + ORIGEN)})
display(tabla_origen.round(4))
print("Diferencia por usar dataset:", round(tabla_origen["promedio"].iloc[1] - tabla_origen["promedio"].iloc[0], 4))

**Decisión 3: `dataset`.** Usarlo no ayuda: el F2 promedio baja de 0.8380 a 0.8316 (-0.0064), y el árbol es el que más lo resiente. De todos modos **no lo uso como variable**, porque el hospital de origen no es una característica del paciente y no serviría con un paciente de un hospital nuevo.

### 2.4 Hiperparámetros con GridSearchCV

Para cada modelo busco los hiperparámetros con `GridSearchCV` sobre entrenamiento, con la validación cruzada `cv` de 5 partes y F2 como métrica. Los valores que pruebo son:

- **Regresión logística:** `C` (qué tanto se frenan los coeficientes) de 0.01 a 100, con y sin `class_weight="balanced"` (que da más peso a la clase con menos casos).
- **KNN:** número de vecinos impar de 1 a 31, con votos iguales o ponderados por cercanía, y distancia de Manhattan o euclidiana.
- **Árbol de decisión:** profundidad máxima, hojas mínimas, criterio (`gini` o `entropy`) y `class_weight`.

In [ ]:
columnas_finales = NUM_FINAL + CAT_FINAL

cuadriculas = {
    "Regresión logística": {"modelo__C": [0.01, 0.1, 1, 10, 100], "modelo__class_weight": [None, "balanced"]},
    "KNN": {"modelo__n_neighbors": list(range(1, 32, 2)), "modelo__weights": ["uniform", "distance"], "modelo__p": [1, 2]},
    "Árbol de decisión": {"modelo__max_depth": [2, 3, 4, 5, 6, 8], "modelo__min_samples_leaf": [1, 5, 10, 20],
                          "modelo__criterion": ["gini", "entropy"], "modelo__class_weight": [None, "balanced"]},
}

busquedas = {}
for nombre, cuadricula in cuadriculas.items():
    busqueda = GridSearchCV(armar_modelo(nombre, ESCALAR[nombre], categoricas=CAT_FINAL, numericas=NUM_FINAL), cuadricula, cv=cv, scoring=f2)
    busqueda.fit(X_train[columnas_finales], y_train)
    busquedas[nombre] = busqueda

resumen = pd.DataFrame({
    nombre: {"combinaciones probadas": len(b.cv_results_["params"]), "F2 en validación cruzada (media)": b.best_score_,
             "desviación": b.cv_results_["std_test_score"][b.best_index_]}
    for nombre, b in busquedas.items()
}).T
display(resumen.round(4))
for nombre, b in busquedas.items():
    print(f"{nombre}:", {k.replace("modelo__", ""): v for k, v in b.best_params_.items()})

**Qué veo.** Las mejores configuraciones son:

- **Regresión logística:** `C = 1` y sin `class_weight`. F2 de 0.8541 en validación cruzada.
- **KNN:** 3 vecinos, distancia de Manhattan (`p = 1`) y votos iguales. F2 de 0.8388.
- **Árbol de decisión:** `gini`, profundidad 2 y mínimo de 20 pacientes por hoja. F2 de 0.8345.

La desviación entre las 5 partes es de 0.0562 en regresión logística y de 0.0522 en el árbol, y de 0.0292 en KNN. Esas desviaciones son más grandes que la diferencia entre modelos, así que con la validación cruzada los tres quedan casi empatados. Se nota también que KNN eligió solo 3 vecinos, un valor muy chico que suele ajustarse demasiado al entrenamiento, y que el árbol eligió una profundidad de 2, el más simple de los que probé.

### 2.5 Entrenamiento contra validación y prueba: ¿hay sobreajuste?

Evalúo cada modelo ya ajustado en las tres partes. Un modelo que sobreajusta saca mucho más en entrenamiento que en validación. **La elección entre los tres se hace con la validación**, no con la prueba. Regla escrita antes de ver los números: elijo el modelo con mayor F2 en validación, y si dos quedan a menos de 0.02, me quedo con el que tenga mejor recall.

In [ ]:
filas = []
for nombre, busqueda in busquedas.items():
    modelo = busqueda.best_estimator_
    for parte, (Xp, yp) in {"entrenamiento": (X_train, y_train), "validación": (X_val, y_val), "prueba": (X_test, y_test)}.items():
        filas.append({"modelo": nombre, "parte": parte, **metricas(modelo, Xp[columnas_finales], yp)})
tabla_metricas = pd.DataFrame(filas)
display(tabla_metricas.pivot(index="modelo", columns="parte", values="F2")[["entrenamiento", "validación", "prueba"]].round(4).rename_axis(columns="F2"))
display(tabla_metricas.round(4).set_index(["modelo", "parte"]))

validacion = tabla_metricas[tabla_metricas["parte"] == "validación"].set_index("modelo")
mejor_f2 = validacion["F2"].max()
cercanos = validacion[validacion["F2"] >= mejor_f2 - 0.02]
elegido = cercanos["recall"].idxmax()
print("Modelos a menos de 0.02 de F2 del mejor en validación:", list(cercanos.index))
print("Modelo elegido (regla escrita antes):", elegido)

**Qué veo.** Las tres partes del F2 son:

| Modelo | Entrenamiento | Validación | Prueba |
|---|---|---|---|
| Regresión logística | 0.8787 | 0.8624 | 0.8960 |
| KNN | 0.9126 | 0.8429 | 0.8577 |
| Árbol de decisión | 0.8697 | 0.8349 | 0.8872 |

**Sobreajuste.** KNN es el que más baja de entrenamiento a validación (de 0.9126 a 0.8429, y su AUC va de 0.9585 a 0.8122). Eso cuadra con sus 3 vecinos: casi memoriza el entrenamiento. La regresión logística es la más pareja (de 0.8787 a 0.8624). El árbol de profundidad 2 baja poco en F2, pero tiene una precision baja en las tres partes (0.7266, 0.6870 y 0.7218): detecta bien a los enfermos pero marca a muchos sanos como enfermos.

**Elección.** Con la regla escrita antes, el mejor F2 en validación es el de la regresión logística (0.8624). KNN queda a menos de 0.02 (0.8429), pero el árbol no (0.8349). Entre la regresión logística y KNN, la regresión logística tiene mejor recall en validación (0.8725 contra 0.8627). **Me quedo con la regresión logística.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (nombre, busqueda) in zip(axes, busquedas.items()):
    matriz = confusion_matrix(y_test, busqueda.best_estimator_.predict(X_test[columnas_finales]))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["sin enfermedad", "con enfermedad"], yticklabels=["sin enfermedad", "con enfermedad"])
    ax.set_title(nombre + (" (elegido)" if nombre == elegido else ""), fontsize=10)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

filas = []
for nombre, busqueda in busquedas.items():
    tn, fp, fn, tp = confusion_matrix(y_test, busqueda.best_estimator_.predict(X_test[columnas_finales])).ravel()
    filas.append({"modelo": nombre, "enfermos que se dicen sanos (falsos negativos)": int(fn), "sanos que se dicen enfermos (falsos positivos)": int(fp),
                  "enfermos detectados": int(tp), "sanos bien clasificados": int(tn)})
display(pd.DataFrame(filas).set_index("modelo"))

**Qué veo.** En prueba hay 184 pacientes: 102 enfermos y 82 sanos. Los errores de cada modelo son:

| Modelo | Enfermos que se dicen sanos | Sanos que se dicen enfermos |
|---|---|---|
| Regresión logística | 9 | 18 |
| KNN | 14 | 17 |
| Árbol de decisión | 6 | 37 |

El árbol es el que deja pasar menos enfermos (6), pero a cambio marca como enfermos a 37 de los 82 sanos, casi la mitad. La regresión logística deja pasar 9 y se equivoca con 18 sanos. KNN es el peor en enfermos que se le escapan (14).

Hay que decir que el árbol sacó el mejor recall en prueba (0.9412), mejor que la regresión logística (0.9118), pero yo elegí con la validación y no con la prueba. Elegir ahora al árbol porque la prueba me gusta más sería hacer trampa. Además, en este problema un falso positivo tiene un costo real: un estudio extra, angustia y gasto para una persona sana.

### 2.6 El modelo elegido por dentro

In [ ]:
modelo_elegido = busquedas[elegido].best_estimator_
nombres = modelo_elegido.named_steps["preprocesador"].get_feature_names_out()
paso = modelo_elegido.named_steps["modelo"]

if elegido == "Regresión logística":
    importancia = pd.Series(paso.coef_[0], index=nombres).sort_values(key=abs, ascending=False).head(10)
    titulo = "Los 10 coeficientes más grandes (positivo: sube la probabilidad de enfermedad)"
elif elegido == "Árbol de decisión":
    importancia = pd.Series(paso.feature_importances_, index=nombres).sort_values(ascending=False).head(10)
    titulo = "Importancia de las 10 variables más usadas por el árbol"
    print(export_text(paso, feature_names=list(nombres), max_depth=3))
else:
    importancia = None
    titulo = ""

fbs_falta = df["fbs"].isna()
print("Pacientes sin dato de fbs:", int(fbs_falta.sum()), "| de ellos de Switzerland:", int((fbs_falta & (df["dataset"] == "Switzerland")).sum()),
      "| con enfermedad:", round(df.loc[fbs_falta, "target"].mean() * 100, 1), "% (contra", round(df.loc[~fbs_falta, "target"].mean() * 100, 1),
      "% en quienes sí tienen el dato)")

if importancia is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(x=importancia.values, y=importancia.index, color="#8172b3", ax=ax)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(titulo, fontsize=10)
    plt.show()
    display(importancia.round(3).rename("valor").to_frame().T)

**Qué veo.** Los coeficientes más grandes de la regresión logística son:

- `fbs_faltante` (0.936), `cp_asymptomatic` (0.921), `slope_flat` (0.838), `sex_Male` (0.702) y `thal_reversable defect` (0.567), que **suben** la probabilidad de enfermedad.
- `fbs_no` (-0.88), `cp_atypical angina` (-0.798), `slope_downsloping` (-0.553) y `thal_normal` (-0.548), que la **bajan**.

Los coeficientes de `sex_Male` y `sex_Female` salen casi iguales con signo contrario (0.702 y -0.698) porque no quité una columna de cada categoría, así que se reparten el mismo efecto.

Lo que sí tiene sentido médico: un dolor de pecho sin síntomas (`asymptomatic`), ser hombre, un defecto reversible en el estudio `thal` y una pendiente plana del ST suben el riesgo, y una angina atípica o un `thal` normal lo bajan.

**Lo que no es médico.** El coeficiente más grande es `fbs_faltante`, o sea, que el dato de glucosa en ayunas no esté. A 90 pacientes les falta `fbs`, 75 de ellos son de Suiza, y de esos 90 el 84.4% tiene enfermedad, contra el 52.2% en quienes sí tienen el dato. Entonces el modelo está usando "que falte este dato" como pista del hospital, y no una señal clínica. Es una limitación real de cómo traté los faltantes.

### 2.7 ¿Qué tan estable es la elección? Veinte repartos

Con 184 pacientes en validación y en prueba, un solo reparto puede favorecer a cualquiera. Repito todo el proceso con 20 repartos distintos (`random_state` de 0 a 19): separo, busco los hiperparámetros con `GridSearchCV` solo con el entrenamiento de ese reparto, elijo con la validación y mido en la prueba. Esto no sirve para escoger el modelo, solo para ver cuánto varía.

In [ ]:
filas = []
for semilla in range(20):
    X_r, X_t, y_r, y_t = train_test_split(X, y, test_size=0.20, stratify=y, random_state=semilla)
    X_tr, X_va, y_tr, y_va = train_test_split(X_r, y_r, test_size=0.25, stratify=y_r, random_state=semilla)
    cv_local = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    por_modelo = {}
    for nombre, cuadricula in cuadriculas.items():
        busqueda = GridSearchCV(armar_modelo(nombre, ESCALAR[nombre], categoricas=CAT_FINAL, numericas=NUM_FINAL), cuadricula, cv=cv_local, scoring=f2)
        busqueda.fit(X_tr[columnas_finales], y_tr)
        por_modelo[nombre] = {"validación": fbeta_score(y_va, busqueda.predict(X_va[columnas_finales]), beta=2),
                              "recall (validación)": recall_score(y_va, busqueda.predict(X_va[columnas_finales])),
                              **{f"{k} (prueba)": v for k, v in metricas(busqueda, X_t[columnas_finales], y_t).items()}}
    val = pd.Series({n: m["validación"] for n, m in por_modelo.items()})
    cerca = val[val >= val.max() - 0.02]
    elegido_local = max(cerca.index, key=lambda n: por_modelo[n]["recall (validación)"])
    for nombre, m in por_modelo.items():
        filas.append({"reparto": semilla, "modelo": nombre, "elegido": nombre == elegido_local, **m})
repartos = pd.DataFrame(filas)

resumen = repartos.groupby("modelo")[["validación", "F2 (prueba)", "recall (prueba)", "precision (prueba)", "accuracy (prueba)", "AUC (prueba)"]].agg(["mean", "min", "max"]).round(3)
display(resumen)
print("Veces que cada modelo fue el elegido en validación, de 20 repartos:")
display(repartos[repartos["elegido"]].groupby("modelo").size().rename("veces elegido").to_frame().T)
print("Promedio en prueba del modelo que la regla elegía en cada reparto:")
display(repartos[repartos["elegido"]][["F2 (prueba)", "recall (prueba)", "precision (prueba)", "accuracy (prueba)", "AUC (prueba)"]].mean().round(3).to_frame().T)
mejor_en_prueba = repartos.loc[repartos.groupby("reparto")["F2 (prueba)"].idxmax()].groupby("modelo").size().rename("veces con mejor F2 en prueba").to_frame().T
display(mejor_en_prueba)

**Qué veo.** Con 20 repartos, el F2 promedio en prueba es de 0.857 para la regresión logística (de 0.802 a 0.912), 0.856 para KNN (de 0.812 a 0.907) y 0.830 para el árbol (de 0.734 a 0.893). La regresión logística y KNN quedan prácticamente empatados. El árbol queda atrás sobre todo por su precision: 0.740, contra 0.821 de la regresión logística y 0.828 de KNN, con un recall parecido (0.859, 0.867 y 0.863). Lo mismo pasa con el AUC (0.815 contra 0.885 y 0.879).

La regla de elección que escribí escogió al árbol 10 veces, a la regresión logística 6 y a KNN 4. Eso pasa porque cuando dos modelos quedan a menos de 0.02, la regla desempata por recall, y por lo visto el árbol ganó ese desempate con frecuencia. Pero al medir en prueba, el árbol es el que peor sale. Si sigo la regla en cada reparto, el modelo que queda elegido saca en prueba un F2 promedio de 0.856, un recall de 0.882 y una precision de 0.775.

En la prueba, el modelo con mejor F2 fue KNN en 8 repartos, la regresión logística en 6 y el árbol en 6. Entonces, entre la regresión logística y KNN no hay un ganador claro, y el reparto 42 del notebook (0.8960 de F2 para la regresión logística) salió **mejor que el promedio**.

### 2.8 El origen de los datos: entrenar con tres fuentes y probar con la cuarta

Los pacientes vienen de cuatro hospitales y las columnas no se midieron igual en todos. Para ver qué tanto cambia el resultado con el origen, entreno el modelo elegido (con sus hiperparámetros) con tres fuentes y lo pruebo en la que quedó fuera.

In [ ]:
parametros_elegidos = {k.replace("modelo__", ""): v for k, v in busquedas[elegido].best_params_.items()}
filas = []
for fuente in df["dataset"].unique():
    entrena = df["dataset"] != fuente
    modelo = armar_modelo(elegido, ESCALAR[elegido], parametros_elegidos, categoricas=CAT_FINAL, numericas=NUM_FINAL)
    modelo.fit(df.loc[entrena, columnas_finales], df.loc[entrena, "target"])
    Xf, yf = df.loc[~entrena, columnas_finales], df.loc[~entrena, "target"]
    m = metricas(modelo, Xf, yf) if yf.nunique() > 1 else {}
    filas.append({"fuente que se deja fuera": fuente, "pacientes": len(yf), "% con enfermedad": round(yf.mean() * 100, 1), **m})
display(pd.DataFrame(filas).set_index("fuente que se deja fuera").round(3))

**Qué veo.** Cuando el modelo se enfrenta a un hospital que no vio en el entrenamiento, baja:

| Hospital que se deja fuera | Pacientes | % con enfermedad | Accuracy | Recall | AUC |
|---|---|---|---|---|---|
| Cleveland | 303 | 45.9 | 0.746 | 0.906 | 0.874 |
| Hungría | 294 | 36.1 | 0.823 | 0.821 | 0.898 |
| Suiza | 123 | 93.5 | 0.756 | 0.774 | 0.738 |
| VA Long Beach | 200 | 74.5 | 0.735 | 0.779 | 0.741 |

La accuracy se mueve entre 0.735 y 0.823 y el recall entre 0.774 y 0.906, contra un AUC de 0.9077 en la prueba normal. En Suiza y en VA Long Beach el AUC cae a 0.738 y 0.741. La prueba normal mezcla pacientes de los cuatro hospitales, y por eso sale más favorable que lo que pasaría con un hospital nuevo.

## Recomendación

**Recomiendo la regresión logística** con `C = 1`, entrenada con 5 variables numéricas (`age`, `trestbps`, `chol`, `thalch`, `oldpeak`) y 7 categóricas (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal`), con la mediana para los números faltantes, `faltante` como categoría y las numéricas estandarizadas.

**Resultados en prueba** (184 pacientes, 102 con enfermedad):

- Accuracy de 0.8533, recall de 0.9118, precision de 0.8378, F2 de 0.8960 y AUC de 0.9077.
- Deja pasar a 9 de los 102 enfermos y marca como enfermos a 18 de los 82 sanos.

**Por qué ella.**

- Fue la mejor en F2 en validación (0.8624) y la más pareja entre entrenamiento y validación (0.8787 contra 0.8624), es decir, la que menos sobreajusta.
- En 20 repartos queda igual de buena que KNN (0.857 contra 0.856 de F2 en prueba) y mejor que el árbol (0.830).
- Sus coeficientes se pueden leer y la mayoría tiene sentido médico.

**Qué errores comete.** Lo que más me importaba era no decirle sano a un enfermo, y aun así deja pasar 9 de 102. También le dice enfermo a 18 sanos. Con otro umbral podría detectar más enfermos, a cambio de más sanos mal marcados, pero no lo probé.

## Decisiones

| # | Decisión | Opciones | Elección | Evidencia (solo entrenamiento) |
|---|---|---|---|---|
| 1 | `chol` y `trestbps` en 0 | Dejarlos, borrar filas, tratarlos como faltantes | Faltantes | 172 y 1 pacientes, valores imposibles |
| 2 | Duplicados | Borrar o conservar | Conservar | 2 de 920, no hay identificador para distinguirlos |
| 3 | Outliers | Quitarlos o conservarlos | Conservarlos | Son valores posibles y 5 de 6, 3 de 4, 7 de 8 y 10 de 11 tienen enfermedad |
| 4 | Faltantes numéricos y categóricos | Media, mediana, moda, categoría propia | Mediana y categoría `faltante` | Los extremos mueven la media; los faltantes dependen del hospital |
| 5 | `slope` y `thal` | Quitar o dejar | Dejar | F2 promedio 0.838 contra 0.827 |
| 6 | `ca` | Quitar o dejar | Quitar | 0.8366 con ella contra 0.8380 sin ella |
| 7 | `dataset` | Usar o no | No usar | Baja el F2 (-0.0064) y no sirve con hospitales nuevos |
| 8 | Métrica | Accuracy, recall, F2 | F2 | Un falso negativo es peor que un falso positivo |
| 9 | Modelo | Regresión logística, KNN, árbol | Regresión logística | Mejor F2 en validación (0.8624) y menos sobreajuste |

## Limitaciones

- **Los datos de los cuatro hospitales no son iguales.** Faltan columnas completas en algunos, y la proporción de enfermos va del 36.1% al 93.5%. Al dejar un hospital fuera, la accuracy baja hasta 0.735 y el AUC hasta 0.738.
- **El modelo usa una pista que no es médica.** El coeficiente más grande es que falte `fbs`, que identifica al hospital de Suiza.
- **La tabla es una reconstrucción** con los archivos de UCI y no el archivo de Kaggle, así que mis números pueden no coincidir con los de otro equipo.
- **Con 184 pacientes en prueba**, un solo reparto cambia mucho. El F2 de la regresión logística va de 0.802 a 0.912 entre repartos.
- **No es una herramienta clínica.** No probé otros umbrales ni otros modelos, y estos datos tienen décadas de antigüedad.